# Classification Tutorial - Part 2 - Cross-validation

**Objective of this Notebook:**

The goal of this notebook is to familiarize you with **cross-validation** to improve the robustness and reliability of your model.


___

▶️ Import the **libraries** required for this activity


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_circles
from sklearn.model_selection import cross_validate, KFold, GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
import matplotlib.colors as mcolors

___
▶️ **Exécuter le code** de la fonction d'affichage des graphiques

In [ ]:
def afficher_graphique(
    X, y,                        # X = input data, y = associated classes
    titre="",                    # Figure title, default: empty
    xlabel="Feature 1",          # x-axis label, default: 'Feature 1'
    ylabel="Feature 2",          # y-axis label, default: 'Feature 2'
    w0=None, w1=None, w2=None,   # w0 and w=[w1,w2]^T, parameters defining the separating hyperplane
    svm=None,                    # trained SVM output, used to highlight the margin
    support=None,                # matrix containing support vector coordinates (each row = one support vector)
):
    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    
    # Data
    ax.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='blue', label='Class 0')
    ax.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='red', label='Class 1')
    
    # Decision boundary
    if w0 is not None and w1 is not None and w2 is not None:
        x_values = np.array([X[:, 0].min(), X[:, 0].max()])
        y_values = -(w0 + w1 * x_values) / w2
        ax.plot(x_values, y_values, 'g-', label='Decision boundary')
        
    # Support vectors and margin visualization
    if svm is not None and support is not None:
        xx, yy = np.meshgrid(np.linspace(X[:, 0].min(), X[:, 0].max(), 100),
                             np.linspace(X[:, 1].min(), X[:, 1].max(), 100))
        Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, levels=[-1, 0, 1], alpha=0.3, colors=['gray', 'gray', 'gray'])
        ax.scatter(support[:, 0], support[:, 1], s=100,
                   facecolors='none', edgecolors='green', label='Support vectors')

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.suptitle(titre, fontsize=16, y=.95)
    plt.show()


___
▶️ **Import and display the data** to be classified


In [ ]:
# Create the dataset
data = pd.read_csv('data_class_2.csv')
X = data.iloc[:, :2].to_numpy()
y = data.iloc[:, 2].to_numpy()

# Train on all the data
clf = model = SVC(kernel='linear').fit(X, y)

# Display the result
afficher_graphique(X, y, w0=clf.intercept_[0], w1=clf.coef_[0][0], w2=clf.coef_[0][1])


___
## Part 2.1 Simple cross-validation

### 2.1.1 Classical cross-validation


<div><img src="Loupe.jpg" alt="drawing" width="20"/>
<b>Links to the documentation:</b>
</div>

- [Support Vector Machine (SVM)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
- [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)



#### 💻 Code to complete


In [ ]:
# Initialize the model
model = SVC()

# Configure K-Fold Cross-Validation
kf = KFold()

# Run cross-validation and obtain the scores and models
results = cross_validate()

# Extract the scores for each fold
scores = results['test_score']
print("Accuracy for each fold (in percent):", scores.round(4)*100)
print("Mean accuracy:", scores.mean().round(4)*100, "%")
print("Standard deviation of accuracy:", scores.std().round(4)*100, "%")


In [ ]:
# Graphical visualization of the performance obtained on the different folds

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.histplot(scores*100, bins=np.arange(75,100,2.5), kde=True, color='blue')
plt.title('Histogram of cross-validation scores')
plt.xlim((75, 100))
plt.xlabel('Accuracy')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.boxplot(x=scores*100, color='blue')
plt.xlim((75, 100))
plt.title('Box plot of cross-validation scores')
plt.xlabel('Accuracy')

plt.tight_layout()
plt.show()


▶️ **Run the code** of the cross-validation plot display function


In [ ]:
def plot_kfold_distribution(
    X, y, # X = input data, y = associated classes 
    kf    # data split provided by the KFold function
):
    plt.figure(figsize=(10, 6))
    
    for fold, (train_index, test_index) in enumerate(kf.split(X)):
        y_train = np.full(len(train_index), fold)
        y_test = np.full(len(test_index), fold)
        
        plt.scatter(train_index, y_train, c='blue', label='Training' if fold == 0 else "", marker='_', lw=10, s=1)
        plt.scatter(test_index, y_test, c='orange', label='Test' if fold == 0 else "", marker='_', lw=10, s=1)
    
    plt.xlabel('Sample index')
    plt.ylabel('Fold')
    plt.title('Distribution of samples in the different folds')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()


#### 💻 Code to complete


In [ ]:
# Configure K-Fold Cross-Validation
kf_std = KFold() # TO COMPLETE
plot_kfold_distribution() # TO COMPLETE


# Run cross-validation and obtain the scores and models
results_std = cross_validate(model, X, y, cv=kf_std)


# Extract the scores for each fold
scores_std = results_std['test_score']
print("Accuracy for each fold (in percent):", scores_std.round(4)*100)
print("Mean accuracy:", scores_std.mean().round(4)*100, "%")
print("Standard deviation of accuracy:", scores_std.std().round(4)*100, "%")


___
### 2.1.2 Cross-validation with data shuffling before splitting


#### 💻 Code to complete


In [ ]:
# Configure K-Fold Cross-Validation
kf_Shuffle = KFold() # TO COMPLETE
plot_kfold_distribution() # TO COMPLETE

# Run cross-validation and obtain the scores and models
results_Shuffle = cross_validate(model, X, y, cv=kf_Shuffle)


# Extract the scores for each fold
scores_Shuffle = results_Shuffle['test_score']
print("Accuracy for each fold (in percent):", scores_Shuffle.round(4)*100)
print("Mean accuracy:", scores_Shuffle.mean().round(4)*100, "%")
print("Standard deviation of accuracy:", scores_Shuffle.std().round(4)*100, "%")

___
## Part 2.2 Stratified cross-validation

▶️ **Run the code** of the stratified cross-validation plot display function


In [ ]:
def plot_stratified_kfold_distribution(
    X, y, # X = input data, y = associated classes 
    cv,   # output of the StratifiedKFold function
):
    n_splits = cv.n_splits
    
    fig, ax = plt.subplots(figsize=(10, 6))
    cmap_cv = mcolors.ListedColormap(["blue", "orange"])
    cmap_data = mcolors.ListedColormap(['red', 'green'])
    
    for ii, (tr, tt) in enumerate(cv.split(X, y)):
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1
        indices[tr] = 0
        ax.scatter(
            range(len(indices)),
            [ii + 0.5] * len(indices),
            c=indices,
            marker="_",
            lw=10,
            cmap=cmap_cv,
            vmin=-0.2,
            vmax=1.2,
        )
    ax.scatter(
        range(len(X)), [ii + 1.5] * len(X), c=y, marker="_", lw=10, cmap=cmap_data
    )
    yticklabels = list(range(n_splits)) + ["class"]
    ax.set(
        yticks=np.arange(n_splits + 1) + 0.5,
        yticklabels=yticklabels,
        xlabel="Sample index",
        ylabel="Fold",
        ylim=[n_splits + 1.2, -0.2],
        xlim=[0, len(X)],
    )
    ax.set_title("{}".format(type(cv).__name__), fontsize=15)
    plt.show()


#### 💻 Code to complete


In [ ]:
# Configure K-Fold Cross-Validation
kf_Strat = StratifiedKFold() # TO COMPLETE
plot_stratified_kfold_distribution() # TO COMPLETE

kf_Strat = StratifiedKFold(n_splits=10, shuffle=False)
plot_stratified_kfold_distribution(X, y, kf_Strat)


# Run cross-validation and obtain the scores and models
results_Strat = cross_validate(model, X, y, cv=kf_Strat)


# Extract the scores for each fold
scores_Strat = results_Strat['test_score']
print("Accuracy for each fold (in percent):", scores_Strat.round(4)*100)
print("Mean accuracy:", scores_Strat.mean().round(4)*100, "%")
print("Standard deviation of accuracy:", scores_Strat.std().round(4)*100, "%")
